In [8]:
import polars as pl 
import pandas as pd
import os
import pandas as pd
import scipy.sparse as sp
import numpy as np
from pathlib import Path
import re
import sys
from zebra_dev.utils import preprocess, plotting, training 
from zebra_dev.models import dilated_baseline_model

lifelong_dir = Path("../data/lifelong/processed/to_train")
embryo_dir = Path("../data/embryo/processed/to_train")

lifelong_files = sorted(lifelong_dir.glob("*.parquet"))
embryo_files = sorted(embryo_dir.glob("*.parquet"))

print(lifelong_files)
print(embryo_files)

[PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l0_q0__seqs.parquet'), PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l1_q0__seqs.parquet'), PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l1_q1__seqs.parquet'), PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_raw_l0_q0__seqs.parquet'), PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_raw_l1_q0__seqs.parquet'), PosixPath('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_raw_l1_q1__seqs.parquet')]
[PosixPath('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l0_q0__seqs.parquet'), PosixPath('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l0_q1__seqs.parquet'), PosixPath('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l1_q0__seqs.parquet'), PosixPath('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l1_q1__seqs.parqu

In [4]:
lifelong_dfs = [pl.read_parquet(f) for f in lifelong_files]
embryo_dfs   = [pl.read_parquet(f) for f in embryo_files]

In [3]:
lifelong_dir = Path("../data/lifelong/processed/to_train")
embryo_dir = Path("..data/embryo/processed/to_train")

merged_dfs = preprocess.merge_embryo_lifelong_files(lifelong_dir, embryo_dir)

output_dir = Path("../data/merged_embryo2life/")
output_dir.mkdir(exist_ok=True, parents=True)

for pattern, df in merged_dfs.items():
    output_name = pattern.replace('DATASET', 'merged') + '.parquet'
    output_path = output_dir / output_name
    df.write_parquet(output_path)

Found 0 matching file pairs


In [4]:
# load the train_atac_L2k_g11_merged_cpm_l1_q1 file to dataframae
merged_Cpm_l1_q1= pl.read_parquet('../data/merged_embryo2life/train_atac_L2k_g11_merged_cpm_l1_q1__seqs.parquet')
merged_Cpm_l1_q1

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,14_bone,14_cycling cells,14_dermal fibroblast,14_gill cartilage,14_gill progenitor 1,14_gill stroma,14_hyaline cartilage,14_periosteum/tendon/ligament,14_perivascular,14_pillar,14_smooth muscle,14_smooth muscle 2,14_stroma 1,14_teeth,14_tunica media,…,5_gill stroma,5_hyaline cartilage,5_hypoblast,5_periosteum/tendon/ligament,5_perivascular,5_smooth muscle,5_smooth muscle 2,5_stroma 1,5_teeth,60_bone,60_cycling cells,60_dermal fibroblast,60_gill cartilage,60_gill progenitor 1,60_gill progenitor 2,60_gill stroma,60_hyaline cartilage,60_periosteum/tendon/ligament,60_perivascular,60_pillar,60_smooth muscle,60_smooth muscle 2,60_stroma 1,60_teeth,60_tunica media,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,Peak,chromosome,end,peak_id,sequence,start,dataset
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str,i64,i64,str,str,i64,str
4.517199,4.136115,2.934742,4.235682,2.84646,3.813445,3.511894,5.314108,5.139036,1.986546,2.859997,2.653156,0.0,2.837419,4.081522,2.846386,2.834833,2.859676,4.547888,3.503158,2.84927,4.683784,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,6.474885,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.855224,3.45226,2.845114,4.339328,5.053746,"""chr10:10002124-10002624""",10,10002624,"""0""","""TGCTTTCTCATGCAGCAAACACGTGTGCAT…",10002124,"""embryo"""
2.827393,2.478875,4.424094,1.330676,3.996338,2.855775,3.884604,1.883948,3.891707,1.986827,2.048824,5.008417,0.0,3.966897,3.558452,3.995495,4.896295,5.248831,4.070387,2.005437,4.004315,4.347085,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,4.784791,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.015522,2.204501,3.994541,0.872023,0.609591,"""chr10:10003707-10004207""",10,10004207,"""0""","""TAGCTTTTAACAATAAAATAAACAAACAAA…",10003707,"""embryo"""
4.219712,2.478387,3.563446,3.380321,3.996336,4.325491,4.516629,3.637527,2.827693,3.983323,3.523999,4.191887,4.510599,3.966896,3.913076,3.995493,4.336807,5.290722,4.006738,2.005443,4.004312,4.19743,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,2.220506,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.788177,2.822938,3.994538,1.645249,0.971994,"""chr10:10004747-10005247""",10,10005247,"""0""","""TTTCTGCTCATAGCTGGATCCATTTCCGCT…",10004747,"""embryo"""
4.133026,2.47839,4.134303,3.507416,3.996334,4.555839,3.913254,4.526872,4.456573,3.970854,5.023106,3.909793,0.0,3.966894,4.529378,3.995491,4.336824,4.484913,3.363778,4.45975,4.004308,4.335056,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,3.636795,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.502574,3.913933,3.994535,4.322604,5.034261,"""chr10:10008047-10008547""",10,10008547,"""0""","""ATCTCTAATAAAGCTGTGGTACTTAAATAA…",10008047,"""embryo"""
2.186927,2.478394,3.520511,1.330678,3.996332,2.545397,1.175765,4.013578,6.745711,1.98691,2.04883,3.558654,0.059938,3.966884,2.178789,3.995489,4.336732,2.554547,2.846,3.464218,4.004305,1.223859,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,2.030926,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.972088,1.673812,3.994531,2.03505,0.608867,"""chr10:10009662-10010162""",10,10010162,"""0""","""CTAACATTTGAG

In [4]:
merged_Cpm_l1_q0 = pl.read_parquet('../data/merged_embryo2life/train_atac_L2k_g11_merged_cpm_l1_q0__seqs.parquet')
merged_Cpm_l1_q0

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,14_bone,14_cycling cells,14_dermal fibroblast,14_gill cartilage,14_gill progenitor 1,14_gill stroma,14_hyaline cartilage,14_periosteum/tendon/ligament,14_perivascular,14_pillar,14_smooth muscle,14_smooth muscle 2,14_stroma 1,14_teeth,14_tunica media,…,5_gill stroma,5_hyaline cartilage,5_hypoblast,5_periosteum/tendon/ligament,5_perivascular,5_smooth muscle,5_smooth muscle 2,5_stroma 1,5_teeth,60_bone,60_cycling cells,60_dermal fibroblast,60_gill cartilage,60_gill progenitor 1,60_gill progenitor 2,60_gill stroma,60_hyaline cartilage,60_periosteum/tendon/ligament,60_perivascular,60_pillar,60_smooth muscle,60_smooth muscle 2,60_stroma 1,60_teeth,60_tunica media,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,Peak,chromosome,end,peak_id,sequence,start,dataset
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,str,i64,i64,str,str,i64,str
5.736278,4.528056,5.269069,5.570145,0.0,5.468067,5.373735,5.825353,5.990973,0.0,0.0,4.74258,0.0,0.0,5.662337,0.0,0.0,5.307293,5.720568,4.825613,0.0,5.999563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,5.418043,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.211814,4.865353,0.0,5.949215,6.171589,"""chr10:10002124-10002624""",10,10002624,"""0""","""TGCTTTCTCATGCAGCAAACACGTGTGCAT…",10002124,"""embryo"""
4.805037,0.0,5.682519,0.0,0.0,5.205603,5.437842,0.0,5.590126,0.0,0.0,6.045309,0.0,0.0,5.506392,0.0,5.400784,6.149087,5.581296,0.0,0.0,5.818902,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,5.129559,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.327934,0.0,"""chr10:10003707-10004207""",10,10004207,"""0""","""TAGCTTTTAACAATAAAATAAACAAACAAA…",10003707,"""embryo"""
5.563784,0.0,5.459051,5.276048,0.0,5.604928,5.582504,4.836142,5.312269,5.517022,4.253798,5.683096,4.686146,0.0,5.622125,0.0,0.0,6.182027,5.577286,0.0,0.0,5.758648,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,4.798491,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.492745,4.302743,0.0,5.535282,5.139282,"""chr10:10004747-10005247""",10,10005247,"""0""","""TTTCTGCTCATAGCTGGATCCATTTCCGCT…",10004747,"""embryo"""
5.534709,0.0,5.596153,5.336432,0.0,5.736506,5.449131,5.383111,5.75084,5.510648,5.543799,5.587443,0.0,0.0,5.899639,0.0,0.0,5.878319,5.442904,5.615714,0.0,5.808381,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,4.962676,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.39824,5.242149,0.0,5.943246,6.161794,"""chr10:10008047-10008547""",10,10008547,"""0""","""ATCTCTAATAAAGCTGTGGTACTTAAATAA…",10008047,"""embryo"""
0.0,0.0,5.419276,0.0,0.0,5.069403,4.898797,5.042924,6.83868,0.0,0.0,5.471278,0.0,0.0,0.0,0.0,0.0,5.062457,5.379211,4.686465,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,4.760256,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.21907,0.0,0.0,5.621454,0.0,"""chr10:10009662-10010162""",10,10010162,"""0""","""CTAACATTTGAGCGCTATTGAGCACAGTCT…",10009662,"""embryo"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [5]:
embryo_cpm_l1_q1 = pl.read_parquet('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l1_q1__seqs.parquet')
embryo_cpm_l1_q1

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,18_UND,18_YSL,18_anterior/posterior axis,18_blastomere,18_central nervous system,18_digestive system,18_erythroid lineage cell,18_forebrain,18_immature eye,18_integument,18_lateral plate mesoderm,18_mesenchyme cell,18_musculature system,18_neural crest,18_neural keel,18_neural stem cell,18_periderm/epidermis,18_primary neuron,18_segmental plate,24_UND,24_YSL,24_anterior/posterior axis,24_blastomere,24_central nervous system,24_digestive system,24_erythroid lineage cell,24_forebrain,24_immature eye,24_integument,24_mesenchyme cell,24_musculature system,24_neural stem cell,24_periderm/epidermis,24_primary neuron,24_segmental plate,3_blastomere,5_EVL,5_YSL/presumptive endoderm,5_blastomere,5_epiblast,5_hypoblast,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,chromosome,start,end,sequence,Peak
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,i64,str,str
4.517199,4.136115,2.934742,4.235682,2.84646,3.813445,3.511894,5.314108,5.139036,1.986546,2.859997,2.653156,0.0,2.837419,4.081522,2.846386,2.834833,2.859676,4.547888,3.503158,2.84927,4.683784,2.049451,2.646331,5.262817,2.844696,5.00411,2.219585,4.47381,3.49211,2.81495,2.416506,2.840463,4.125342,0.881157,2.84134,2.830724,3.528206,4.50596,4.523752,3.910585,5.893196,2.223711,2.835823,2.845876,4.333672,3.628183,4.231074,1.988503,0.949647,0.874832,5.402155,0.966167,2.841897,3.497396,4.365794,2.56361,1.585413,2.220145,1.978799,2.846958,3.553658,6.474885,2.855224,3.45226,2.845114,4.339328,5.053746,10,10002124,10002624,"""TGCTTTCTCATGCAGCAAACACGTGTGCAT…","""chr10:10002124-10002624"""
2.827393,2.478875,4.424094,1.330676,3.996338,2.855775,3.884604,1.883948,3.891707,1.986827,2.048824,5.008417,0.0,3.966897,3.558452,3.995495,4.896295,5.248831,4.070387,2.005437,4.004315,4.347085,3.626947,4.370085,4.556149,4.442966,3.451148,2.219611,4.456847,2.47584,4.512313,4.099038,3.983544,3.005777,6.070393,3.985388,3.899322,3.528272,4.136377,4.348903,2.800356,3.817694,3.361018,3.911029,3.995523,3.349825,1.976885,2.22567,3.382647,2.230284,4.315087,3.367419,2.843117,3.553942,4.505831,3.706253,3.79878,2.649046,2.556603,4.668756,3.996323,2.826841,4.784791,1.015522,2.204501,3.994541,0.872023,0.609591,10,10003707,10004207,"""TAGCTTTTAACAATAAAATAAACAAACAAA…","""chr10:10003707-10004207"""
4.219712,2.478387,3.563446,3.380321,3.996336,4.325491,4.516629,3.637527,2.827693,3.983323,3.523999,4.191887,4.510599,3.966896,3.913076,3.995493,4.336807,5.290722,4.006738,2.005443,4.004312,4.19743,3.778844,3.971438,2.867913,4.442474,3.128263,2.219614,2.200867,4.569386,4.095775,4.2223,3.98354,1.138708,4.193235,3.985384,5.008261,4.537722,4.562623,3.790499,2.551879,3.075577,3.361021,3.911017,3.995519,4.35221,1.976888,4.876536,4.003508,4.001812,4.563818,3.530739,3.148768,3.53052,3.546034,4.33957,5.488233,2.817165,2.846778,4.594962,3.996322,2.81108,2.220506,3.788177,2.822938,3.994538,1.645249,0.971994,10,10004747,10005247,"""TTTCTGCTCATAGCTGGATCCATTTCCGCT…","""chr10:10004747-10005247"""
4.133026,2.47839,4.134303,3.507416,3.996334,4.555839,3.913254,4.526872,4.456573,3.970854,5.023106,3.909793,0.0,3.966894,4.529378,3.995491,4.336824,4.484913,3.363778,4.45975,4.004308,4.335056,4.209813,2.646187,4.196352,4.442616,3.568568,2.219618,4.084423,4.475182,2.817715,3.377143,3.983535,3.823359,4.208941,3.985382,3.899311,2.557168,3.698869,5.256132,4.230854,4.361297,3.361025,3.911015,3.995518,4.668564,4.431123,4.22558

In [6]:
embryo_cpm_l0_q0= pl.read_parquet('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l0_q0__seqs.parquet')
embryo_cpm_l0_q0

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,18_UND,18_YSL,18_anterior/posterior axis,18_blastomere,18_central nervous system,18_digestive system,18_erythroid lineage cell,18_forebrain,18_immature eye,18_integument,18_lateral plate mesoderm,18_mesenchyme cell,18_musculature system,18_neural crest,18_neural keel,18_neural stem cell,18_periderm/epidermis,18_primary neuron,18_segmental plate,24_UND,24_YSL,24_anterior/posterior axis,24_blastomere,24_central nervous system,24_digestive system,24_erythroid lineage cell,24_forebrain,24_immature eye,24_integument,24_mesenchyme cell,24_musculature system,24_neural stem cell,24_periderm/epidermis,24_primary neuron,24_segmental plate,3_blastomere,5_EVL,5_YSL/presumptive endoderm,5_blastomere,5_epiblast,5_hypoblast,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,chromosome,start,end,sequence,Peak
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,str,str
308.908779,91.578373,193.234956,261.472088,0.0,236.001547,214.666778,337.78078,398.803589,0.0,0.0,113.729858,0.0,0.0,286.820594,0.0,0.0,200.803213,304.078077,123.662895,0.0,402.252615,128.850548,0.0,361.076006,0.0,126.509765,0.0,211.81953,0.0,75.800644,131.769667,0.0,244.155533,0.0,0.0,0.0,0.0,263.869771,241.910956,158.24941,567.056917,0.0,0.0,0.0,95.629722,165.328569,307.361303,143.472023,0.0,0.0,410.468382,0.0,296.240232,212.440708,398.695937,0.0,126.988439,92.66299,54.067453,0.0,143.928579,224.437406,182.426514,128.716695,0.0,382.452325,477.946383,10,10002124,10002624,"""TGCTTTCTCATGCAGCAAACACGTGTGCAT…","""chr10:10002124-10002624"""
121.124031,0.0,292.688292,0.0,0.0,181.290701,228.945369,0.0,266.769321,0.0,0.0,421.128313,0.0,0.0,245.260905,0.0,220.580126,467.28972,264.415346,0.0,0.0,335.602395,216.86728,148.219513,266.897402,0.0,94.747726,0.0,205.806331,0.0,240.206159,257.529547,0.0,183.293156,481.174969,0.0,0.0,0.0,218.201973,207.344125,105.15336,310.55054,0.0,0.0,0.0,0.0,0.0,0.0,285.729575,204.139516,389.184201,248.563898,217.308009,327.814378,370.37037,338.195414,0.0,139.458676,96.601572,109.99864,0.0,133.159416,167.942572,0.0,0.0,0.0,205.01198,0.0,10,10003707,10004207,"""TAGCTTTTAACAATAAAATAAACAAACAAA…","""chr10:10003707-10004207"""
259.807742,0.0,233.874333,194.595395,0.0,270.76241,264.736281,124.982393,201.809974,247.892746,69.372182,292.85768,107.434465,0.0,275.476408,0.0,0.0,482.971968,263.353099,0.0,0.0,315.919594,220.726189,111.164635,0.0,0.0,91.794424,0.0,0.0,263.921879,186.033002,271.501444,0.0,29.922202,260.579546,0.0,260.45058,229.166732,280.983152,141.003948,97.107976,259.727605,0.0,0.0,0.0,108.979948,0.0,527.42616,350.555574,311.388644,457.805585,261.139274,236.971766,324.249304,232.034388,391.331118,646.203554,141.067371,101.823805,109.326022,0.0,132.567425,120.327219,241.923228,72.902238,0.0,252.479315,169.593302,10,10004747,10005247,"""TTTCTGCTCATAGCTGGATCCATTTCCGCT…","""chr10:10004747-10005247"""
252.33409,0.0,268.388006,206.770064,0.0,308.979453,231.555875,216.698401,313.45483,246.31129,254.647313,266.05181,0.0,0.0,363.905602,0.0,0.0,356.208444,230.112476,273.709554,0.0,332.079584,250.773108,0.0,188.810672,0.0,99.538267,0.0,120.729204,217.438574,76.195276,195.64601,0.0,226.507533,262.131423,0.0,0.0,0.0,179.401662,342.618294,177.179495,360.988199,0.0,0.0,0.0,290.44438,251.214576,304.924531,371.439775,529.80397,243.077952,244.76291,294.998326,366.959376,0.0,345.128358,0.0,166.429343,105.982112,0.0,0.0,148.543472,141.97584,2

In [7]:
embryo_cpm_l1_q0= pl.read_parquet('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l1_q0__seqs.parquet')
embryo_cpm_l1_q0

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,18_UND,18_YSL,18_anterior/posterior axis,18_blastomere,18_central nervous system,18_digestive system,18_erythroid lineage cell,18_forebrain,18_immature eye,18_integument,18_lateral plate mesoderm,18_mesenchyme cell,18_musculature system,18_neural crest,18_neural keel,18_neural stem cell,18_periderm/epidermis,18_primary neuron,18_segmental plate,24_UND,24_YSL,24_anterior/posterior axis,24_blastomere,24_central nervous system,24_digestive system,24_erythroid lineage cell,24_forebrain,24_immature eye,24_integument,24_mesenchyme cell,24_musculature system,24_neural stem cell,24_periderm/epidermis,24_primary neuron,24_segmental plate,3_blastomere,5_EVL,5_YSL/presumptive endoderm,5_blastomere,5_epiblast,5_hypoblast,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,chromosome,start,end,sequence,Peak
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,str,str
5.736278,4.528056,5.269069,5.570145,0.0,5.468067,5.373735,5.825353,5.990973,0.0,0.0,4.74258,0.0,0.0,5.662337,0.0,0.0,5.307293,5.720568,4.825613,0.0,5.999563,4.866384,0.0,5.891854,0.0,4.848193,0.0,5.360445,0.0,4.341213,4.888616,0.0,5.501893,0.0,0.0,0.0,0.0,5.579238,5.492695,5.070472,6.342222,0.0,0.0,0.0,4.570886,5.113965,5.731272,4.973086,0.0,0.0,6.019732,0.0,5.694541,5.363359,5.990704,0.0,4.85194,4.539703,4.008559,0.0,4.976241,5.418043,5.211814,4.865353,0.0,5.949215,6.171589,10,10002124,10002624,"""TGCTTTCTCATGCAGCAAACACGTGTGCAT…","""chr10:10002124-10002624"""
4.805037,0.0,5.682519,0.0,0.0,5.205603,5.437842,0.0,5.590126,0.0,0.0,6.045309,0.0,0.0,5.506392,0.0,5.400784,6.149087,5.581296,0.0,0.0,5.818902,5.383886,5.005418,5.590604,0.0,4.561717,0.0,5.331783,0.0,5.485652,5.55501,0.0,5.216528,6.178307,0.0,0.0,0.0,5.389994,5.339191,4.664885,5.741562,0.0,0.0,0.0,0.0,0.0,0.0,5.65854,5.32369,5.966619,5.519715,5.385907,5.795493,5.9172,5.826576,0.0,4.944913,4.580894,4.709518,0.0,4.899029,5.129559,0.0,0.0,0.0,5.327934,0.0,10,10003707,10004207,"""TAGCTTTTAACAATAAAATAAACAAACAAA…","""chr10:10003707-10004207"""
5.563784,0.0,5.459051,5.276048,0.0,5.604928,5.582504,4.836142,5.312269,5.517022,4.253798,5.683096,4.686146,0.0,5.622125,0.0,0.0,6.182027,5.577286,0.0,0.0,5.758648,5.401443,4.719968,0.0,0.0,4.530387,0.0,0.0,5.579435,5.231285,5.607644,0.0,3.431474,5.566738,0.0,5.566245,5.438804,5.641847,4.955855,4.586069,5.563476,0.0,0.0,0.0,4.700298,0.0,6.269903,5.862368,5.744248,6.128627,5.568876,5.472152,5.784592,5.451186,5.972106,6.472661,4.956301,4.633017,4.70344,0.0,4.894606,4.798491,5.492745,4.302743,0.0,5.535282,5.139282,10,10004747,10005247,"""TTTCTGCTCATAGCTGGATCCATTTCCGCT…","""chr10:10004747-10005247"""
5.534709,0.0,5.596153,5.336432,0.0,5.736506,5.449131,5.383111,5.75084,5.510648,5.543799,5.587443,0.0,0.0,5.899639,0.0,0.0,5.878319,5.442904,5.615714,0.0,5.808381,5.528528,0.0,5.246027,0.0,4.610538,0.0,4.801799,5.386505,4.346338,5.281405,0.0,5.427183,5.572654,0.0,0.0,0.0,5.195186,5.839531,5.182791,5.891612,0.0,0.0,0.0,5.674849,5.53028,5.723338,5.920075,6.274393,5.497488,5.504367,5.690354,5.907973,0.0,5.84681,0.0,5.120561,4.672662,0.0,0.0,5.007587,4.962676,5.39824,5.242149,0.0,5.943246,6.161794,10,10008047,10008547,"""ATCTCTAATAAAGCTGTGGTACTTAAATAA…","""chr10:10008047-10008547"""
0.0,0.0,5.419276,0.0,0.0,5.069403,4.898797,5.042924,6.83868,0.0,0.0,5.471278,0.0,0.0,0.0,0.0,0.0,5.062457,5.379211,4.686465,0.0,0.0,4.245337,0.0,4.535476,0.0,4.540025,3.731297,0.0,0.0,5.3436

In [8]:
embryo_cpm_l0_q1= pl.read_parquet('../data/embryo/processed/to_train/train_atac_L2k_g11_embryo_cpm_l0_q1__seqs.parquet')
embryo_cpm_l0_q1

10_UND,10_YSL,10_anterior/posterior axis,10_lateral plate mesoderm,10_mesenchyme cell,10_neural crest,10_neural keel,10_periderm/epidermis,10_segmental plate,12_UND,12_YSL,12_anterior/posterior axis,12_central nervous system,12_integument,12_lateral plate mesoderm,12_mesenchyme cell,12_musculature system,12_neural crest,12_neural keel,12_periderm/epidermis,12_primary neuron,12_segmental plate,18_UND,18_YSL,18_anterior/posterior axis,18_blastomere,18_central nervous system,18_digestive system,18_erythroid lineage cell,18_forebrain,18_immature eye,18_integument,18_lateral plate mesoderm,18_mesenchyme cell,18_musculature system,18_neural crest,18_neural keel,18_neural stem cell,18_periderm/epidermis,18_primary neuron,18_segmental plate,24_UND,24_YSL,24_anterior/posterior axis,24_blastomere,24_central nervous system,24_digestive system,24_erythroid lineage cell,24_forebrain,24_immature eye,24_integument,24_mesenchyme cell,24_musculature system,24_neural stem cell,24_periderm/epidermis,24_primary neuron,24_segmental plate,3_blastomere,5_EVL,5_YSL/presumptive endoderm,5_blastomere,5_epiblast,5_hypoblast,6_EVL,6_YSL/presumptive endoderm,6_blastomere,6_epiblast,6_hypoblast,chromosome,start,end,sequence,Peak
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,i64,str,str
227.962051,181.65477,103.516212,190.998688,98.3694,154.247849,132.303711,355.770782,316.733673,55.719524,100.409851,87.162895,0.0,97.040047,171.626236,98.359222,96.671623,100.357681,236.296783,130.979279,98.78019,257.566986,63.489147,86.344009,336.50473,98.111839,293.992493,71.455704,217.887878,129.318176,93.989967,77.42968,97.478424,179.519379,21.259735,97.599632,96.094063,134.953476,225.407364,229.585907,161.46199,424.843445,72.070816,96.802299,98.287987,198.545059,143.377823,189.978348,55.901749,22.956522,20.641773,364.502563,24.534348,97.682556,130.099228,205.860443,83.18924,41.198456,71.534241,54.977695,98.445259,139.26358,704.243042,99.66452,124.32618,98.17202,199.830444,310.26947,10,10002124,10002624,"""TGCTTTCTCATGCAGCAAACACGTGTGCAT…","""chr10:10002124-10002624"""
95.616241,79.235184,208.574951,36.067772,166.684616,99.750671,157.435806,51.546467,158.357208,55.741547,63.397991,295.455383,0.0,162.674118,140.109375,166.502426,281.460052,331.411438,170.558029,57.730427,167.894455,201.499451,143.144165,206.716766,238.571167,211.128372,124.196167,71.459335,214.570099,78.930168,226.927902,174.649734,164.425781,106.268944,502.839661,164.745728,159.568802,134.962204,181.716492,201.888367,92.33287,154.903915,118.945732,161.541016,166.508453,117.747116,54.806286,72.371887,121.27993,73.092323,195.206741,119.552574,97.860031,139.319031,225.371353,146.949966,152.025726,86.645851,82.433983,254.066406,166.681564,95.529045,268.721252,25.731497,69.309021,166.314835,20.385611,14.365602,10,10003707,10004207,"""TAGCTTTTAACAATAAAATAAACAAACAAA…","""chr10:10003707-10004207"""
187.574112,79.178749,140.99704,121.030502,166.684158,197.013733,227.825317,144.48172,95.656937,164.383301,134.277161,183.121017,226.567322,162.673721,161.885971,166.502029,199.191772,346.492798,168.301773,57.731102,167.893845,184.267792,149.147339,163.441315,101.650185,211.001038,109.487442,71.459831,68.812607,242.405533,173.997543,188.054321,164.425034,27.956739,183.410934,164.745102,295.402496,233.501099,240.50621,150.795532,81.906487,108.503418,118.946167,161.539658,166.507629,202.643448,54.80669,276.371674,167.738297,167.499191,240.872833,135.349258,111.244118,135.314224,137.922943,199.89006,371.480225,94.250351,98.417282,250.216446,166.681244,93.515778,71.585907,150.388138,95.017433,166.314163,42.245949,25.201117,10,10004747,10005247,"""TTTCTGCTCATAGCTGGATCCATTTCCGCT…","""chr10:10004747-10005247"""
181.011124,79.179443,181.276382,131.624207,166.683807,238.482239,161.921143,230.470505,

In [7]:
lifelong_cpm_l1_q1 = pl.read_parquet('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l1_q1__seqs.parquet')
lifelong_cpm_l1_q1

14_teeth,14_dermal fibroblast,14_perivascular,14_periosteum/tendon/ligament,14_gill progenitor 1,14_stroma 1,14_bone,14_smooth muscle,14_hyaline cartilage,14_pillar,14_gill cartilage,14_gill stroma,14_tunica media,14_cycling cells,14_smooth muscle 2,210_teeth,210_gill progenitor 2,210_gill cartilage,210_gill progenitor 1,210_perivascular,210_periosteum/tendon/ligament,210_dermal fibroblast,210_smooth muscle 2,210_pillar,210_smooth muscle,210_stroma 1,210_tunica media,210_gill stroma,210_bone,210_cycling cells,2_cycling cells,2_gill progenitor 1,2_periosteum/tendon/ligament,2_hyaline cartilage,2_stroma 1,3_hyaline cartilage,3_gill progenitor 1,…,3_stroma 1,3_smooth muscle 2,3_tunica media,5_teeth,5_gill progenitor 1,5_stroma 1,5_periosteum/tendon/ligament,5_perivascular,5_cycling cells,5_dermal fibroblast,5_hyaline cartilage,5_bone,5_gill stroma,5_smooth muscle,5_gill cartilage,5_smooth muscle 2,60_gill progenitor 2,60_gill progenitor 1,60_stroma 1,60_teeth,60_dermal fibroblast,60_smooth muscle 2,60_smooth muscle,60_periosteum/tendon/ligament,60_gill stroma,60_pillar,60_cycling cells,60_perivascular,60_gill cartilage,60_bone,60_tunica media,60_hyaline cartilage,chromosome,start,end,sequence,peak_id
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,i64,str,str
0.168097,0.260924,0.097243,0.094158,0.064181,0.031402,0.069372,0.021527,0.776579,0.406317,0.024237,0.820144,0.052172,1.551402,0.088595,0.027683,0.015602,0.004908,0.001683,0.027593,0.140338,0.114465,0.0,0.254545,0.034479,0.006647,0.042311,0.039012,0.230254,0.081447,0.071577,0.009583,0.009399,0.752378,0.586175,0.010498,0.052207,…,0.064282,0.04466,0.059472,0.086336,0.05412,0.001875,0.062045,0.009706,0.104836,0.0187,0.113918,0.27159,0.084545,0.32879,0.088391,0.085437,0.005096,0.013278,0.015202,0.039449,0.566827,0.039107,0.898124,0.002925,0.133361,0.02328,0.033932,0.004492,0.062034,0.76812,0.229693,0.033201,1,1168,1395,"""TCAAAGAGCTGTTGCAGCATGAGTGGAAAA…","""chr1-1168-1395"""
0.25792,0.328231,0.127745,0.306451,0.04362,0.11682,1.024627,0.096635,0.038708,0.033949,0.352835,0.0,0.052177,0.162359,0.460289,0.027697,0.062985,0.031567,0.004254,0.00592,0.044919,0.0,0.921116,0.305456,0.03448,0.030184,0.042311,0.039011,0.0,0.186218,0.111306,0.059538,0.08128,0.250615,0.0,0.036377,0.018937,…,0.328924,0.044648,0.059493,0.24063,0.271933,0.015931,0.088741,0.192031,0.234048,0.078874,0.046049,0.085426,0.193127,0.067633,0.206137,0.44207,0.021718,0.146227,0.015201,0.0,0.630894,0.039114,0.032115,0.010572,0.133332,0.00121,0.331054,0.015398,0.518629,0.030063,0.229593,0.2642,1,3061,3345,"""TCTGGGTTACAAGTTTTAGAAGAATTAATA…","""chr1-3061-3345"""
0.180822,0.037679,0.260638,0.073737,0.895887,0.39259,1.257913,0.096636,0.038685,0.033948,0.141214,0.0,0.052177,0.688626,0.460287,0.101623,1.506816,0.62228,0.064573,0.097635,0.108995,0.0,1.673793,0.019944,0.302587,0.281114,0.042306,0.039012,0.0,2.147595,0.06343,0.158294,0.885401,0.250976,0.0,0.110066,0.018697,…,0.043516,0.044648,0.059493,0.112198,0.338007,0.001877,0.085146,0.214859,0.435193,0.492164,0.046049,0.085426,0.433555,0.067632,0.458462,0.442068,1.670518,0.146201,0.237651,0.119537,0.128864,0.039114,0.032115,0.010573,0.365519,0.020048,0.171534,0.069439,0.149506,0.030063,0.229592,0.290622,1,4198,4561,"""AATGACTCATTAGTATTCGTGTTTTTGGAC…","""chr1-4198-4561"""
1.220496,1.173946,0.770668,1.00503,1.445618,0.741676,0.903448,0.096637,0.971953,0.406805,0.456205,0.0,0.052171,1.177061,0.460282,0.867843,0.748733,0.677463,1.210626,0.759687,0.548693,0.769017,1.454772,0.355413,1.230562,0.504082,1.96928,0.461846,0.0,0.410985,0.697222,0.423944,0.929176,1.576445,0.0,0.03595,0.693746,…,0.503968,0.531585,0.059491,0.793191,0.958124,0.793262,0.743743,1.15201,0.838098,1.081788,0.311615,1.270804,0.433552,0.067632,0.458461,0.442064,0.963868,0.264385,0.9

In [8]:
lifelong_cpm_l0_q0 = pl.read_parquet('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l0_q0__seqs.parquet')
lifelong_cpm_l0_q0

14_teeth,14_dermal fibroblast,14_perivascular,14_periosteum/tendon/ligament,14_gill progenitor 1,14_stroma 1,14_bone,14_smooth muscle,14_hyaline cartilage,14_pillar,14_gill cartilage,14_gill stroma,14_tunica media,14_cycling cells,14_smooth muscle 2,210_teeth,210_gill progenitor 2,210_gill cartilage,210_gill progenitor 1,210_perivascular,210_periosteum/tendon/ligament,210_dermal fibroblast,210_smooth muscle 2,210_pillar,210_smooth muscle,210_stroma 1,210_tunica media,210_gill stroma,210_bone,210_cycling cells,2_cycling cells,2_gill progenitor 1,2_periosteum/tendon/ligament,2_hyaline cartilage,2_stroma 1,3_hyaline cartilage,3_gill progenitor 1,…,3_stroma 1,3_smooth muscle 2,3_tunica media,5_teeth,5_gill progenitor 1,5_stroma 1,5_periosteum/tendon/ligament,5_perivascular,5_cycling cells,5_dermal fibroblast,5_hyaline cartilage,5_bone,5_gill stroma,5_smooth muscle,5_gill cartilage,5_smooth muscle 2,60_gill progenitor 2,60_gill progenitor 1,60_stroma 1,60_teeth,60_dermal fibroblast,60_smooth muscle 2,60_smooth muscle,60_periosteum/tendon/ligament,60_gill stroma,60_pillar,60_cycling cells,60_perivascular,60_gill cartilage,60_bone,60_tunica media,60_hyaline cartilage,chromosome,start,end,sequence,peak_id
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,i64,str,str
0.307579,0.399774,0.224029,0.223661,0.142655,0.136352,0.0,0.0,1.154196,0.418246,0.0,0.0,0.0,4.828865,0.0,0.063516,0.0,0.0,0.0,0.0,0.294835,0.0,0.0,0.28869,0.0,0.0,0.0,0.0,0.281117,0.0,0.169993,0.0,0.0,1.225106,0.0,0.045711,0.114586,…,0.143444,0.0,0.0,0.0,0.126392,0.0,0.132372,0.0,0.195714,0.0,0.12563,0.370985,0.0,0.0,0.0,0.0,0.0,0.128642,0.0,0.0,1.307918,0.0,2.410399,0.0,0.0,0.156614,0.0,0.0,0.083792,1.6688,0.0,0.0,1,1168,1395,"""TCAAAGAGCTGTTGCAGCATGAGTGGAAAA…","""chr1-1168-1395"""
0.423126,0.486597,0.275852,0.483911,0.109693,0.275589,1.845182,0.0,0.0,0.0,0.449638,0.0,0.0,0.0,0.0,0.063516,0.0,0.0,0.0,0.0,0.121595,0.0,1.641991,0.416093,0.0,0.0,0.0,0.0,0.0,0.0,0.209357,0.095688,0.06617,0.0,0.0,0.109874,0.036578,…,0.620943,0.0,0.0,0.307432,0.393695,0.058511,0.163165,0.318965,0.372838,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.476595,0.0,0.0,1.434971,0.0,0.0,0.0,0.0,0.0,0.661738,0.078879,0.957103,0.0,0.0,0.339936,1,3061,3345,"""TCTGGGTTACAAGTTTTAGAAGAATTAATA…","""chr1-3061-3345"""
0.323022,0.079409,0.452762,0.200092,1.422328,0.623487,2.63128,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.223963,3.021887,0.799109,0.175153,0.184213,0.235911,0.0,4.58509,0.0,0.398181,0.552019,0.0,0.0,0.0,8.894107,0.159325,0.231842,1.448425,0.0,0.0,0.234126,0.030302,…,0.094334,0.0,0.0,0.142543,0.467533,0.0,0.159509,0.345168,0.631096,0.668779,0.0,0.0,0.0,0.0,0.0,0.0,4.224368,0.476479,0.65377,0.33294,0.226914,0.0,0.0,0.0,0.668156,0.143311,0.370289,0.285511,0.31448,0.0,0.0,0.353952,1,4198,4561,"""AATGACTCATTAGTATTCGTGTTTTTGGAC…","""chr1-4198-4561"""
2.542466,2.374374,1.207708,1.779676,3.304365,1.201677,1.545772,0.0,1.578082,0.418246,0.623172,0.0,0.0,2.698677,0.0,1.446454,1.15728,0.871624,1.921434,1.297425,0.837092,1.356348,3.503854,0.486901,2.943897,0.905138,6.133416,0.687836,0.0,0.0,1.035631,0.55256,1.553399,4.619345,0.0,0.108656,1.303499,…,0.929269,0.936724,0.0,1.255576,1.715115,1.505108,1.178934,2.49967,1.518539,2.249056,0.40952,3.136703,0.0,0.0,0.0,0.0,1.890725,0.671083,2.12339,1.202756,2.94381,0.0,0.0,2.043361,0.0,1.610482,0.777975,0.873373,2.294017,0.0,1.986397,3.322629,1,5995,6563,"""TCAGGTGTGAATGCTGTTTTCTGTTTCAAA…","""chr1-5995-6563"""
13.791148,15.326517,14.565765,13.696332,18.518408,14.011777,17.356337,16.306665,14.439282,16.966019,9.692588,11.274161,9.366359,21.407749,0.0,10.407068,10.564945,11.973659,9.696199,10.196934,13.708504,12.750184,14.916473,11.498051,9.595533,8.791641,2.605022,13.743148,8.630569,30.286591,18.700445,18.242762,20.245464,17.303837,10.818267,12.953285,13.422

In [10]:
lifelong_cpm_l1_q0 = pl.read_parquet('../data/lifelong/processed/to_train/train_atac_L2k_g11_lifelong_cpm_l1_q0__seqs.parquet')
lifelong_cpm_l1_q0

14_teeth,14_dermal fibroblast,14_perivascular,14_periosteum/tendon/ligament,14_gill progenitor 1,14_stroma 1,14_bone,14_smooth muscle,14_hyaline cartilage,14_pillar,14_gill cartilage,14_gill stroma,14_tunica media,14_cycling cells,14_smooth muscle 2,210_teeth,210_gill progenitor 2,210_gill cartilage,210_gill progenitor 1,210_perivascular,210_periosteum/tendon/ligament,210_dermal fibroblast,210_smooth muscle 2,210_pillar,210_smooth muscle,210_stroma 1,210_tunica media,210_gill stroma,210_bone,210_cycling cells,2_cycling cells,2_gill progenitor 1,2_periosteum/tendon/ligament,2_hyaline cartilage,2_stroma 1,3_hyaline cartilage,3_gill progenitor 1,…,3_stroma 1,3_smooth muscle 2,3_tunica media,5_teeth,5_gill progenitor 1,5_stroma 1,5_periosteum/tendon/ligament,5_perivascular,5_cycling cells,5_dermal fibroblast,5_hyaline cartilage,5_bone,5_gill stroma,5_smooth muscle,5_gill cartilage,5_smooth muscle 2,60_gill progenitor 2,60_gill progenitor 1,60_stroma 1,60_teeth,60_dermal fibroblast,60_smooth muscle 2,60_smooth muscle,60_periosteum/tendon/ligament,60_gill stroma,60_pillar,60_cycling cells,60_perivascular,60_gill cartilage,60_bone,60_tunica media,60_hyaline cartilage,chromosome,start,end,sequence,peak_id
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64,i64,str,str
0.268178,0.336311,0.202148,0.201847,0.133354,0.127823,0.0,0.0,0.767418,0.349421,0.0,0.0,0.0,1.762822,0.0,0.061581,0.0,0.0,0.0,0.0,0.258383,0.0,0.0,0.253626,0.0,0.0,0.0,0.0,0.247733,0.0,0.156998,0.0,0.0,0.799805,0.0,0.044697,0.108483,…,0.134045,0.0,0.0,0.0,0.119019,0.0,0.124314,0.0,0.178744,0.0,0.118343,0.315529,0.0,0.0,0.0,0.0,0.0,0.121015,0.0,0.0,0.836346,0.0,1.226829,0.0,0.0,0.145497,0.0,0.0,0.080466,0.981629,0.0,0.0,1,1168,1395,"""TCAAAGAGCTGTTGCAGCATGAGTGGAAAA…","""chr1-1168-1395"""
0.352856,0.39649,0.243614,0.394681,0.104083,0.243408,1.045627,0.0,0.0,0.0,0.371314,0.0,0.0,0.0,0.0,0.061581,0.0,0.0,0.0,0.0,0.114752,0.0,0.971533,0.347902,0.0,0.0,0.0,0.0,0.0,0.0,0.190088,0.091382,0.064073,0.0,0.0,0.104247,0.035925,…,0.483008,0.0,0.0,0.268065,0.331958,0.056864,0.151145,0.276847,0.31688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.389739,0.0,0.0,0.889935,0.0,0.0,0.0,0.0,0.0,0.507864,0.075922,0.671465,0.0,0.0,0.292622,1,3061,3345,"""TCTGGGTTACAAGTTTTAGAAGAATTAATA…","""chr1-3061-3345"""
0.279919,0.076414,0.373467,0.182398,0.884729,0.484576,1.289585,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.202094,1.391751,0.587292,0.161398,0.169078,0.211808,0.0,1.720101,0.0,0.335172,0.439557,0.0,0.0,0.0,2.291939,0.147838,0.208511,0.895445,0.0,0.0,0.210363,0.029852,…,0.090146,0.0,0.0,0.133256,0.383583,0.0,0.147997,0.296519,0.489252,0.512092,0.0,0.0,0.0,0.0,0.0,0.0,1.653334,0.38966,0.503057,0.287387,0.204502,0.0,0.0,0.0,0.511719,0.133929,0.315022,0.251157,0.273441,0.0,0.0,0.303028,1,4198,4561,"""AATGACTCATTAGTATTCGTGTTTTTGGAC…","""chr1-4198-4561"""
1.264823,1.21621,0.791955,1.022334,1.45963,0.789219,0.934434,0.0,0.947046,0.349421,0.484382,0.0,0.0,1.307975,0.0,0.894639,0.768848,0.626806,1.072075,0.831789,0.608184,0.857113,1.504934,0.396694,1.372169,0.644554,1.96479,0.523447,0.0,0.0,0.710806,0.439905,0.937425,1.726215,0.0,0.103149,0.834429,…,0.657141,0.660998,0.0,0.813405,0.998834,0.918332,0.778836,1.252669,0.923679,1.178365,0.343249,1.419899,0.0,0.0,0.0,0.0,1.061507,0.513472,1.138919,0.789709,1.372147,0.0,0.0,1.112963,0.0,0.959535,0.575475,0.627741,1.192108,0.0,1.094068,1.463864,1,5995,6563,"""TCAGGTGTGAATGCTGTTTTCTGTTTCAAA…","""chr1-5995-6563"""
2.694029,2.792791,2.745074,2.687598,2.971358,2.708835,2.909975,2.851092,2.736915,2.888482,2.369551,2.507496,2.338566,3.109407,0.0,2.434233,2.447978,2.562921,2.369889,2.41564,2.688426,2.621052,2.767355,2.525573,2.360433,2.281529,1.282328,2.690778,2.264942,3.44319,2.980641,2.957135,3.056144,2.907111,2.469646,2.635715,2.668817,…,2.350582,3.2011

In [ ]:
# create training data for **EMBRYO**
out_embryo_dir = training.prepare_training_from_df(
    df=embryo_cpm_l1_q1,
    pattern="train_atac_L2k_g11_embryo_cpm_l1_q1__seqs",
    output_dir="../data/embryo/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        # or "random"
    holdout_chroms=None,            # or a list of specified ["1","2"] 
    channel_first=True
)
print("Saved to:", out_embryo_dir)

# create training data for **LIFELONG**
out_lifelong_dir = training.prepare_training_from_df(
    df=lifelong_cpm_l1_q1,
    pattern="train_atac_L2k_g11_lifelong_cpm_l1_q1__seqs",
    output_dir="../data/lifelong/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        
    holdout_chroms=None,       
    channel_first=True
)
print("Saved to:", out_lifelong_dir)





Saved to: ../../data/lifelong/training/train_atac_L2k_g11_lifelong_cpm_l1_q1__seqs/80_20_chrom_split
Saved to: ../../data/merged_embryo2life/training/train_atac_L2k_g11_merged_cpm_l1_q1__seqs/80_20_chrom_split


In [ ]:
## split random - mixed chromosomes 
# create training data for **EMBRYO**
out_embryo_dir = training.prepare_training_from_df(
    df=embryo_cpm_l1_q1,
    pattern="train_atac_L2k_g11_embryo_cpm_l1_q1__seqs",
    output_dir="../data/embryo/training/",
    test_size=0.2,
    random_state=42,
    split_mode="random",        # or "random"
    holdout_chroms=None,            # or a list of specified ["1","2"] 
    channel_first=True
)
print("Saved to:", out_embryo_dir)

# create training data for **LIFELONG**
out_lifelong_dir = training.prepare_training_from_df(
    df=lifelong_cpm_l1_q1,
    pattern="train_atac_L2k_g11_lifelong_cpm_l1_q1__seqs",
    output_dir="../data/lifelong/training/",
    test_size=0.2,
    random_state=42,
    split_mode="random",        
    holdout_chroms=None,       
    channel_first=True
)


Saved to: ../data/embryo/training/train_atac_L2k_g11_embryo_cpm_l1_q1__seqs/80_20_random


In [ ]:
out_embryo_dir_l0 = training.prepare_training_from_df(
    df=embryo_cpm_l0_q0,
    pattern="train_atac_L2k_g11_embryo_cpm_l0_q0__seqs",
    output_dir="../data/embryo/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        # or "random"
    holdout_chroms=None,            # or a list of specified ["1","2"] 
    channel_first=True
)

# create training data for **LIFELONG**
out_lifelong_dir_l0_q0 = training.prepare_training_from_df(
    df=lifelong_cpm_l0_q0,
    pattern="train_atac_L2k_g11_lifelong_cpm_l0_q0__seqs",
    output_dir="../data/lifelong/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        
    holdout_chroms=None,       
    channel_first=True
)



In [ ]:
out_embryo_dir_l1q0 = training.prepare_training_from_df(
    df=embryo_cpm_l1_q0,
    pattern="train_atac_L2k_g11_embryo_cpm_l1_q0__seqs",
    output_dir="../data/embryo/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        # or "random"
    holdout_chroms=None,            # or a list of specified ["1","2"] 
    channel_first=True
)

# create training data for **LIFELONG**
out_lifelong_dir_l0_q0 = training.prepare_training_from_df(
    df=lifelong_cpm_l1_q0,
    pattern="train_atac_L2k_g11_lifelong_cpm_l1_q0__seqs",
    output_dir="../data/lifelong/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        
    holdout_chroms=None,       
    channel_first=True
)


In [ ]:
out_embryo_dir_l1q0 = training.prepare_training_from_df(
    df=embryo_cpm_l0_q1,
    pattern="train_atac_L2k_g11_embryo_raw_l0_q1__seqs",
    output_dir="../data/embryo/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        # or "random"
    holdout_chroms=None,            # or a list of specified ["1","2"] 
    channel_first=True
)

In [ ]:
# create training data for **EMBRYO + LIFELONG**
out_merged_dir = training.prepare_training_from_df(
    df=merged_Cpm_l1_q1,
    pattern="train_atac_L2k_g11_merged_cpm_l1_q1__seqs",
    output_dir="../data/merged_embryo2life/training/",
    test_size=0.2,
    random_state=42,
    split_mode="chromosome",        
    holdout_chroms=None,            
    channel_first=True
)
print("Saved to:", out_merged_dir)

# bundle = load_split_folder(out_dir)
# bundle["X_train"].shape, bundle["Y_train"].shape, bundle["meta"]



AttributeError: module 'zebra_dev.utils.preprocess' has no attribute 'prepare_training_from_df'

### Create train data for RNA


In [2]:
rna_embryo = pd.read_csv('../data/embryo/processed/rna_data_enriched_psd.csv', index_col=0)
print(rna_embryo.shape)
rna_embryo.head()

(26544, 63)


,18_anterior/posterior axis,12_neural keel,5_hypoblast,24_anterior/posterior axis,18_erythroid lineage cell,5_epiblast,18_lateral plate mesoderm,10_anterior/posterior axis,12_anterior/posterior axis,18_primary neuron,...,12_integument,24_immature eye,18_integument,24_periderm/epidermis,18_digestive system,24_central nervous system,24_primary neuron,18_central nervous system,24_erythroid lineage cell,24_segmental plate
rpl13a,5.857117,5.824826,3.413838,6.313237,6.463578,2.643843,4.899677,6.112449,5.610301,5.522120,...,5.727920,6.026385,5.590376,4.972049,4.869890,5.234285,3.818170,5.585933,5.804019,5.487815
khdrbs1a,6.122264,6.536900,6.499459,6.308115,5.110812,6.003681,5.216568,6.766642,6.098865,5.602649,...,3.000000,5.336023,5.045748,3.306467,4.587463,5.118621,3.831246,5.729262,3.331637,5.100458
apoeb,3.069851,4.656436,4.655129,5.135443,2.887316,4.500979,3.287546,5.262438,4.416610,1.978619,...,3.169925,1.805192,6.859398,0.312531,2.877444,1.864325,0.351518,3.186347,0.397705,4.151361
cfl1,4.765997,4.585853,3.999817,5.109143,4.746344,3.242275,4.030919,4.876013,4.445181,4.565309,...,0.000000,4.676847,3.772863,0.661758,3.890680,3.951784,3.086633,4.238597,3.150882,3.872751
polr2d,1.697659,2.096149,1.172371,1.975166,1.579638,0.691351,1.968964,2.115665,1.782080,1.109292,...,2.321928,1.743021,1.325530,0.416848,0.792481,1.262022,0.484377,1.702378,0.665596,1.363183


#### add gene coordinates


In [3]:
gtf_path = "../data/genome/Danio_rerio.GRCz11.111.chr.gtf"
fasta_path = "../data/genome/Danio_rerio.GRCz11.dna.primary_assembly.fa"
gene_info = preprocess.extract_gene_info_from_gtf(gtf_path)
print(gene_info.shape)
gene_info.head()
# merge the gene info with rna_embryo based on gene_name which is the index of rna_embryo
rna_embryo_annot = rna_embryo.merge(gene_info, left_index=True, right_on='gene_name', how='left')
rna_embryo_annot = rna_embryo_annot.set_index('gene_name')
# remove rows with nas in the columns chr, start, end
rna_embryo_annot = rna_embryo_annot.dropna(subset=['chrom', 'start', 'end'])

rna_embryo_annot.head()
# transform start and end to int
rna_embryo_annot['start'] = rna_embryo_annot['start'].astype(int)
rna_embryo_annot['end'] = rna_embryo_annot['end'].astype(int)
rna_embryo_annot['tss_position'] = rna_embryo_annot['tss_position'].astype(int)
rna_embryo_annot.head()
# discard rows with nas in the columns chr, start, end
rna_embryo_annot = rna_embryo_annot.dropna(subset=['chrom', 'start', 'end'])
# drom rows where the chrom is MT
rna_embryo_annot = rna_embryo_annot[rna_embryo_annot['chrom'] != 'MT']
# convert to integer the chrom column
rna_embryo_annot['chrom'] = rna_embryo_annot['chrom'].astype(int)
rna_embryo_annot.head()

(31954, 7)


,18_anterior/posterior axis,12_neural keel,5_hypoblast,24_anterior/posterior axis,18_erythroid lineage cell,5_epiblast,18_lateral plate mesoderm,10_anterior/posterior axis,12_anterior/posterior axis,18_primary neuron,...,24_primary neuron,18_central nervous system,24_erythroid lineage cell,24_segmental plate,chrom,start,end,tss_position,gene_id,strand
gene_name,,,,,,,,,,,,,,,,,,,,,
rpl13a,5.857117,5.824826,3.413838,6.313237,6.463578,2.643843,4.899677,6.112449,5.610301,5.522120,...,3.818170,5.585933,5.804019,5.487815,17,24615091,24618279,24615091,ENSDARG00000044093,+
khdrbs1a,6.122264,6.536900,6.499459,6.308115,5.110812,6.003681,5.216568,6.766642,6.098865,5.602649,...,3.831246,5.729262,3.331637,5.100458,13,45009957,45022527,45022527,ENSDARG00000052856,-
apoeb,3.069851,4.656436,4.655129,5.135443,2.887316,4.500979,3.287546,5.262438,4.416610,1.978619,...,0.351518,3.186347,0.397705,4.151361,16,23960744,23963461,23960744,ENSDARG00000040295,+
cfl1,4.765997,4.585853,3.999817,5.109143,4.746344,3.242275,4.030919,4.876013,4.445181,4.565309,...,3.086633,4.238597,3.150882,3.872751,14,30795559,30801555,30795559,ENSDARG00000021124,+
polr2d,1.697659,2.096149,1.172371,1.975166,1.579638,0.691351,1.968964,2.115665,1.782080,1.109292,...,0.484377,1.702378,0.665596,1.363183,2,1124512,1129721,1124512,ENSDARG00000076509,+


In [5]:
# only the gene sequence
rna_with_seq = preprocess.add_sequences_to_rna(
    rna_embryo_annot,
    fasta_dir="../data/genome",
    save_dir="../data/embryo/processed/to_train"
)

[✓] Saved to ../data/embryo/processed/to_train/rna_with_gene_seq_ba0.csv


In [6]:
# +/- 1000 bp around the tss
rna_with_seq_ba1k = preprocess.add_sequences_to_rna(
    rna_embryo_annot,
    fasta_dir="../data/genome",
    symmetric_bp=1000,
    save_dir="../data/embryo/processed/to_train"
)

[✓] Saved to ../data/embryo/processed/to_train/rna_with_gene_seq_ba1000.csv


In [10]:
# +/- 1000 bp around the tss
rna_with_seq_b1ka500 = preprocess.add_sequences_to_rna(
    rna_embryo_annot,
    fasta_dir="../data/genome",
    before_bp=1000,
    after_bp=500,
    save_dir="../data/embryo/processed/to_train"
)

[✓] Saved to ../data/embryo/processed/to_train/rna_with_gene_seq_ba1000_500.csv


#### expand sequence tensors to include the the accessibility value of each nucleotide across each gene and its flanking regions

## Multiome


In [9]:
import anndata as ad
multiome_data = ad.read_h5ad(
    "../data/multiome/zf_multiome_atlas_full_ATAC_v1_release.h5ad"
)
multiome_data


AnnData object with n_obs × n_vars = 94562 × 640834
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'nucleosome_percentile', 'TSS.enrichment', 'TSS.percentile', 'nCount_SCT', 'nFeature_SCT', 'global_annotation', 'nCount_peaks_bulk', 'nFeature_peaks_bulk', 'nCount_peaks_celltype', 'nFeature_peaks_celltype', 'nCount_peaks_merged', 'nFeature_peaks_merged', 'SCT.weight', 'peaks_merged.weight', 'nCount_Gene.Activity', 'nFeature_Gene.Activity', 'nCount_peaks_integrated', 'nFeature_peaks_integrated', 'dataset', 'integrated.weight', 'peaks_integrated.weight', 'wsnn_res.0.8', 'seurat_clusters', 'leiden_0.5', 'leiden_0.8', 'leiden_1', 'leiden_1.2', 'leiden_1.5', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'leiden_6', 'leiden_7', 'leiden_8', 'leiden_9', 'leiden_10', 'leiden_0.5_merged', 'leiden_0.8_merged', 'leiden_1_merged', 'leiden_1.2_merged', 'leiden_1.5_merged', 'leiden_2_merged', 'leiden_3_merged', 'leiden_4_merged', 'leiden_5_me

In [ ]:
celltype_col = "annotation_ML_coarse"  #

import scipy.sparse as sp

adata = multiome_data  # your AnnData object

# choose counts matrix
counts = adata.layers["counts"] if "counts" in adata.layers else adata.X

# cell type labels
labels = adata.obs[celltype_col].astype("category")
categories = labels.cat.categories
codes = labels.cat.codes.to_numpy()

# design matrix: cells × celltypes (one-hot)
row = np.arange(adata.n_obs)
col = codes
data = np.ones(adata.n_obs, dtype=np.int8)
design = sp.csr_matrix((data, (row, col)), shape=(adata.n_obs, len(categories)))

# pseudobulk: (celltypes × cells) @ (cells × peaks) → (celltypes × peaks)
pseudobulk = design.T @ counts  # shape: (n_celltypes, n_peaks)

print("pseudobulk shape:", pseudobulk.shape)
# (num_celltypes, num_peaks)



pseudobulk shape: (32, 640834)


In [41]:
# convert to pandas DataFrame (be careful, may be huge!)
pb_df = pd.DataFrame(
    pseudobulk.toarray().T,     # transpose so peaks = rows
    index=adata.var_names,      # peaks
    columns=categories          # celltypes
)

pb_df.iloc[:5, :5]  # preview


,NMPs,PSM,differentiating_neurons,endocrine_pancreas,endoderm
1-32-526,11,28,14,12,12
1-2372-3057,17,52,26,32,22
1-3427-4032,55,154,62,77,34
1-4469-7268,670,1999,371,893,491
1-9541-9969,152,493,80,179,115


In [ ]:

label_col = "annotation_ML_coarse"  # or "global_annotation"
labels = adata.obs[label_col].astype("category")
counts = adata.layers["counts"] if "counts" in adata.layers else adata.X

# Build a sparse design matrix: cells × celltypes
cats = labels.cat.categories
row = np.arange(adata.n_obs)
col = labels.cat.codes.values
data = np.ones(adata.n_obs, dtype=np.int8)
D = sp.csr_matrix((data, (row, col)), shape=(adata.n_obs, len(cats)))

# Pseudobulk: (celltypes × cells) @ (cells × peaks) = celltypes × peaks
PB = D.T @ counts  # sparse × sparse -> sparse

print("pseudobulk shape:", PB.shape)  # (#celltypes, #peaks)
celltypes = np.array(cats)
peaks = np.array(adata.var_names)


pseudobulk shape: (32, 640834)
